# Module 1 — Portfolio Backtester
**Zaeem Hasan | Quant Portfolio Study**

Tickers held: `LITE AMD COHR GOOG MRVL GLD VRT EQIX AVGO SNDR ANET`  
Entry dates: Aug 8 & Aug 26, 2025 (dollar-cost averaged)  
Benchmark: SPY

**What this notebook does:**
1. Loads real transaction data (your actual entries + share counts)
2. Pulls price history via yfinance
3. Computes portfolio value over time, accounting for DCA
4. Benchmarks against SPY on the same capital deployed
5. Outputs key performance metrics: total return, CAGR, Sharpe, max drawdown
6. Saves clean DataFrames for use in Module 2 (Risk/VaR)


In [ ]:
# ── 0. Install dependencies ──────────────────────────────────────────────────
# Run this cell once. Colab already has most of these.
!pip install yfinance pandas numpy matplotlib seaborn --quiet

In [ ]:
# ── 1. Imports ───────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import date
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f9',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False
})

print('Libraries loaded.')

In [ ]:
# ── 2. Transaction ledger ────────────────────────────────────────────────────
# Your real buys, parsed from broker statements.
# Each row = one purchase event.
# shares = dollar amount invested / price at execution

transactions = pd.DataFrame([
    # ── Aug 8 2024 batch ─────────────────────────────────────────────────────
    {'date': '2025-08-08', 'ticker': 'AMD',  'shares': 0.01917, 'cost_basis': 3.32},
    {'date': '2025-08-08', 'ticker': 'ANET', 'shares': 0.02417, 'cost_basis': 3.38},
    {'date': '2025-08-08', 'ticker': 'AVGO', 'shares': 0.01074, 'cost_basis': 3.28},
    {'date': '2025-08-08', 'ticker': 'COHR', 'shares': 0.02491, 'cost_basis': 2.85},
    {'date': '2025-08-08', 'ticker': 'EQIX', 'shares': 0.00365, 'cost_basis': 2.84},
    {'date': '2025-08-08', 'ticker': 'GOOG', 'shares': 0.01386, 'cost_basis': 2.81},
    {'date': '2025-08-08', 'ticker': 'LITE', 'shares': 0.02504, 'cost_basis': 2.85},
    {'date': '2025-08-08', 'ticker': 'MRVL', 'shares': 0.03670, 'cost_basis': 2.83},
    {'date': '2025-08-08', 'ticker': 'SNDR', 'shares': 0.11823, 'cost_basis': 2.85},
    {'date': '2025-08-08', 'ticker': 'VRT',  'shares': 0.02033, 'cost_basis': 2.85},
    {'date': '2025-08-11', 'ticker': 'GLD',  'shares': 0.00871, 'cost_basis': 2.69},
    # ── Aug 26 2024 batch (DCA) ───────────────────────────────────────────────
    {'date': '2025-08-26', 'ticker': 'AMD',  'shares': 0.01457, 'cost_basis': 2.41},
    {'date': '2025-08-26', 'ticker': 'ANET', 'shares': 0.01806, 'cost_basis': 2.42},
    {'date': '2025-08-26', 'ticker': 'AVGO', 'shares': 0.00814, 'cost_basis': 2.42},
    {'date': '2025-08-26', 'ticker': 'COHR', 'shares': 0.02629, 'cost_basis': 2.42},
    {'date': '2025-08-26', 'ticker': 'EQIX', 'shares': 0.00307, 'cost_basis': 2.42},
    {'date': '2025-08-26', 'ticker': 'GLD',  'shares': 0.00779, 'cost_basis': 2.42},
    {'date': '2025-08-26', 'ticker': 'GOOG', 'shares': 0.01169, 'cost_basis': 2.42},
    {'date': '2025-08-26', 'ticker': 'LITE', 'shares': 0.01918, 'cost_basis': 2.42},
    {'date': '2025-08-26', 'ticker': 'MRVL', 'shares': 0.03301, 'cost_basis': 2.42},
    {'date': '2025-08-26', 'ticker': 'SNDR', 'shares': 0.08111, 'cost_basis': 2.01},
    {'date': '2025-08-26', 'ticker': 'VRT',  'shares': 0.01585, 'cost_basis': 2.01},
    # ── Sep 13 2024 — AMD add ─────────────────────────────────────────────────
    {'date': '2025-09-13', 'ticker': 'AMD',  'shares': 0.05214, 'cost_basis': 8.29},
])

transactions['date'] = pd.to_datetime(transactions['date'])

# Aggregate to total shares per ticker (weighted avg cost basis)
positions = (
    transactions
    .groupby('ticker')
    .agg(
        total_shares=('shares', 'sum'),
        total_cost=('cost_basis', 'sum'),
        first_buy=('date', 'min')
    )
    .reset_index()
)
positions['avg_price'] = positions['total_cost'] / positions['total_shares']

print(f"Positions held: {len(positions)}")
print(f"Total capital deployed: ${positions['total_cost'].sum():.2f}")
print()
print(positions[['ticker','total_shares','total_cost','avg_price']].to_string(index=False))

In [ ]:
# ── 3. Pull price history ────────────────────────────────────────────────────
START_DATE = '2025-08-08'
END_DATE   = date.today().strftime('%Y-%m-%d')
TICKERS    = positions['ticker'].tolist() + ['SPY']

print(f"Pulling data: {START_DATE} → {END_DATE}")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
prices = raw['Close'].copy()
prices.index = pd.to_datetime(prices.index)

print(f"\nDays of data: {len(prices)}")
print(f"Missing values per ticker:")
print(prices.isnull().sum())

In [ ]:
# ── 4. Build portfolio value time series ─────────────────────────────────────
# For each trading day, compute: sum(shares_i * price_i)
# Shares only count from their first purchase date onward.

portfolio_tickers = positions['ticker'].tolist()
port_values = pd.Series(index=prices.index, dtype=float)

for day in prices.index:
    daily_value = 0.0
    for _, row in positions.iterrows():
        if day >= row['first_buy']:
            ticker = row['ticker']
            if ticker in prices.columns and not pd.isna(prices.loc[day, ticker]):
                daily_value += row['total_shares'] * prices.loc[day, ticker]
    port_values[day] = daily_value

# Total capital deployed (as of each date)
total_invested = positions['total_cost'].sum()

# Portfolio return series
port_returns = port_values.pct_change().dropna()

print(f"Starting portfolio value (Aug 8):  ${port_values.iloc[0]:.2f}")
print(f"Current portfolio value:           ${port_values.iloc[-1]:.2f}")
print(f"Total invested:                    ${total_invested:.2f}")
print(f"Unrealized P&L:                    ${port_values.iloc[-1] - total_invested:.2f}")

In [ ]:
# ── 5. Benchmark: SPY on same capital ────────────────────────────────────────
# Buy SPY on Aug 8 with the same total dollars deployed.

spy_entry_price = prices.loc[prices.index >= '2025-08-08', 'SPY'].iloc[0]
spy_shares      = total_invested / spy_entry_price
spy_values      = spy_shares * prices['SPY']
spy_returns     = spy_values.pct_change().dropna()

print(f"SPY entry price:  ${spy_entry_price:.2f}")
print(f"SPY shares bought: {spy_shares:.4f}")
print(f"SPY current value: ${spy_values.iloc[-1]:.2f}")

In [ ]:
# ── 6. Performance metrics ───────────────────────────────────────────────────
TRADING_DAYS = 252
RISK_FREE    = 0.045   # 4.5% — approx current HYSA / T-bill rate

def calc_metrics(values, returns, label):
    total_ret   = (values.iloc[-1] / values.iloc[0]) - 1
    days_held   = (values.index[-1] - values.index[0]).days
    cagr        = (1 + total_ret) ** (365 / days_held) - 1
    ann_vol     = returns.std() * np.sqrt(TRADING_DAYS)
    sharpe      = (returns.mean() * TRADING_DAYS - RISK_FREE) / ann_vol
    rolling_max = values.cummax()
    drawdown    = (values - rolling_max) / rolling_max
    max_dd      = drawdown.min()

    return {
        'Label':           label,
        'Total Return':    f"{total_ret:.1%}",
        'CAGR':            f"{cagr:.1%}",
        'Ann. Volatility': f"{ann_vol:.1%}",
        'Sharpe Ratio':    f"{sharpe:.2f}",
        'Max Drawdown':    f"{max_dd:.1%}",
    }

metrics_port = calc_metrics(port_values, port_returns, 'Your Portfolio')
metrics_spy  = calc_metrics(spy_values,  spy_returns,  'SPY (benchmark)')

summary = pd.DataFrame([metrics_port, metrics_spy]).set_index('Label')
print(summary.to_string())

In [ ]:
# ── 7. Plot: Portfolio vs SPY ────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 12))
fig.suptitle('Module 1 — Portfolio Backtest vs SPY\nEntry: Aug 8–26, 2024', fontsize=14, fontweight='bold')

# 7a. Portfolio value over time
ax1 = axes[0]
ax1.plot(port_values.index, port_values, label='Your Portfolio', color='#4A90D9', linewidth=2)
ax1.plot(spy_values.index,  spy_values,  label='SPY (benchmark)', color='#E87040', linewidth=1.5, linestyle='--')
ax1.axhline(total_invested, color='gray', linewidth=1, linestyle=':', label=f'Capital deployed (${total_invested:.2f})')
ax1.set_ylabel('Portfolio Value ($)')
ax1.set_title('Portfolio Value vs SPY')
ax1.legend()
ax1.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.2f}'))

# 7b. Cumulative return %
ax2 = axes[1]
port_cum = (port_values / port_values.iloc[0] - 1) * 100
spy_cum  = (spy_values  / spy_values.iloc[0]  - 1) * 100
ax2.plot(port_cum.index, port_cum, label='Your Portfolio', color='#4A90D9', linewidth=2)
ax2.plot(spy_cum.index,  spy_cum,  label='SPY', color='#E87040', linewidth=1.5, linestyle='--')
ax2.axhline(0, color='gray', linewidth=0.8)
ax2.set_ylabel('Cumulative Return (%)')
ax2.set_title('Cumulative Return')
ax2.legend()
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())

# 7c. Drawdown
ax3 = axes[2]
port_dd = ((port_values - port_values.cummax()) / port_values.cummax()) * 100
spy_dd  = ((spy_values  - spy_values.cummax())  / spy_values.cummax())  * 100
ax3.fill_between(port_dd.index, port_dd, 0, alpha=0.4, color='#4A90D9', label='Your Portfolio')
ax3.fill_between(spy_dd.index,  spy_dd,  0, alpha=0.3, color='#E87040', label='SPY')
ax3.set_ylabel('Drawdown (%)')
ax3.set_title('Drawdown from Peak')
ax3.legend()
ax3.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('module1_backtest.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as module1_backtest.png')

In [ ]:
# ── 8. Per-ticker breakdown ──────────────────────────────────────────────────
ticker_perf = []
for _, row in positions.iterrows():
    t = row['ticker']
    if t not in prices.columns:
        continue
    t_prices = prices[t].dropna()
    t_prices = t_prices[t_prices.index >= row['first_buy']]
    if len(t_prices) < 2:
        continue
    entry_price   = t_prices.iloc[0]
    current_price = t_prices.iloc[-1]
    pct_return    = (current_price / entry_price - 1) * 100
    current_value = row['total_shares'] * current_price
    cost          = row['total_cost']
    pnl           = current_value - cost
    ticker_perf.append({
        'Ticker':         t,
        'Shares':         round(row['total_shares'], 4),
        'Cost ($)':       round(cost, 2),
        'Entry Price':    round(entry_price, 2),
        'Current Price':  round(current_price, 2),
        'Current Val ($)':round(current_value, 2),
        'P&L ($)':        round(pnl, 2),
        'Return (%)':     round(pct_return, 1)
    })

perf_df = pd.DataFrame(ticker_perf).sort_values('Return (%)', ascending=False)
print(perf_df.to_string(index=False))

In [ ]:
# ── 9. Per-ticker bar chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2ecc71' if r >= 0 else '#e74c3c' for r in perf_df['Return (%)']]
bars = ax.bar(perf_df['Ticker'], perf_df['Return (%)'], color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, perf_df['Return (%)']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (1 if val >= 0 else -3),
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

ax.axhline(0, color='gray', linewidth=0.8)
ax.set_ylabel('Return (%)')
ax.set_title('Per-Ticker Return — Entry Aug 2024 to Today')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('module1_per_ticker.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

In [ ]:
# ── 10. Export for Module 2 (Risk/VaR) ──────────────────────────────────────
# Save the daily returns and prices so Module 2 can load them directly.

port_returns_df = pd.DataFrame({
    'portfolio': port_returns,
    'spy':       spy_returns
})

prices[portfolio_tickers].to_csv('module1_prices.csv')
port_returns_df.to_csv('module1_returns.csv')
perf_df.to_csv('module1_per_ticker_perf.csv', index=False)

print('Exported:')
print('  module1_prices.csv         — daily closing prices')
print('  module1_returns.csv        — daily portfolio + SPY returns')
print('  module1_per_ticker_perf.csv — per-ticker summary')
print()
print('Ready for Module 2 (Risk / VaR).')